# Lab 2: GPT from scratch

In this lab, you will dive into the inner workings of the GPT architecture. You will walk through a complete implementation of the architecture in PyTorch, instantiate this implementation with pre-trained weights, and put the resulting model to the test by generating text. At the end of this lab, you will understand the building blocks of the GPT architecture and how they are connected.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

In [ ]:
from dataclasses import dataclass

import torch
import torch.nn as nn

## Part 1: GPT architecture

GPT-2 was first described by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). To faithfully implement the model, one needs to also read the earlier paper by [Radford et al. (2018)](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf). Another important source of information is the code released by OpenAI, which is available on GitHub ([link](https://github.com/openai/gpt-2)).

The GPT architecture is made up of a stack of Transformer blocks. Each block has two main parts: one handles multi-head self-attention, and the other is a feed-forward network. Before these parts do their work, their input undergoes layer normalisation, and residual connections are added to help the model learn more effectively. The input to the architecture is a sequence of token IDs; these are turned into embeddings and augmented with information about the absolute position of each token in the sequence. The output layer converts the internal representations into logit scores for every token in the vocabulary.

### Model configuration

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) present four increasingly larger GPT models based on the same architecture. Here, we will implement the smallest of these, characterised by the following hyperparameters:

In [ ]:
@dataclass
class Config:
    n_vocab: int = 50_257
    n_ctx: int = 1024
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 12

#### 🧩 Task 2.01: Model configuration

Explain the purpose of these hyperparameters. In particular, where does the number 50,257 come from?

**Answer (Task 2.01):**

- **`n_vocab = 50,257`** — size of the vocabulary. GPT-2 uses Byte Pair Encoding (BPE) tokenisation. The number 50,257 comes from: 50,000 BPE merge rules + 256 base byte tokens + 1 special `<|endoftext|>` token = 50,257 total tokens.
- **`n_ctx = 1024`** — the maximum context length (i.e. the maximum number of tokens the model can attend to at once). This is the size of the context window.
- **`n_embd = 768`** — the embedding dimension. Each token is represented as a 768-dimensional vector throughout the model.
- **`n_head = 12`** — number of attention heads in each multi-head attention layer. Each head operates on `768 / 12 = 64` dimensions.
- **`n_layer = 12`** — number of stacked Transformer decoder blocks in the model.

### GELU activation function

We start by implementing the feed-forward network. This is a standard two-layer network with a Gaussian Error Linear Unit (GELU) activation function ([Hendrycks and Gimpel, 2016](https://doi.org/10.48550/arXiv.1606.08415)).

The GELU is a smooth version of the rectified linear unit (ReLU) that weights inputs by their value under the cumulative distribution function of the standard Gaussian. This function is commonly denoted by $\Phi$. For example, $\text{GELU}(0{.}5) = 0{.}5 \cdot \Phi(0{.}5) \approx 0{.}5 \cdot 0{.}6915 = 0{.}3457$ because approximately 69.15% of normally distributed data lies to the left of $0{.}5$.

When GPT-2 was released, computing the GELU exactly was expensive, and the released code therefore used an approximation originally presented by [Page (1977)](https://doi.org/10.2307/2346872). We follow suit here, as we want to create a replica of the original model. However, it is worth mentioning that PyTorch now offers an exact implementation of the GELU so fast that using an approximation is unnecessary.

In [ ]:
def gelu(x):
    return 0.5 * x * (1 + torch.tanh((2 / torch.pi) ** 0.5 * (x + 0.044715 * x**3)))

#### 🎓 Task 2.02: Mathematical properties of the GELU

Find the minimal output value of the (approximated) GELU and the input value for which it yields that output. Use a service such as [WolframAlpha](https://www.wolframalpha.com/) for the necessary derivations. What are the main differences between the GELU and the ReLU?

**Answer (Task 2.02):**

To find the minimum of GELU, we set its derivative to zero. Using WolframAlpha to differentiate:

$$\text{GELU}(x) = 0.5x\left(1 + \tanh\left(\sqrt{\frac{2}{\pi}}\left(x + 0.044715x^3\right)\right)\right)$$

The minimum occurs at approximately **x ≈ -0.751** with a minimum output value of approximately **GELU(-0.751) ≈ -0.169**.

**Key differences between GELU and ReLU:**

| Property | ReLU | GELU |
|---|---|---|
| Formula | `max(0, x)` | `x · Φ(x)` |
| Smoothness | Not differentiable at x=0 | Smooth and differentiable everywhere |
| Negative outputs | Always 0 for x < 0 | Can produce small negative outputs (min ≈ -0.169) |
| Near-zero behaviour | Hard cutoff at 0 | Smooth transition; inputs near 0 are attenuated probabilistically |
| Gradient for x < 0 | Zero (dead neuron problem) | Non-zero near 0 (mitigates dying neurons) |

The GELU's smooth, probabilistic nature means it doesn't suffer as much from the "dying ReLU" problem, where neurons with negative inputs stop learning entirely. This makes GELU better suited for deep models like GPT-2.

In [ ]:
# Numerically verify the GELU minimum
import torch
x_vals = torch.linspace(-2, 0, 10000)
y_vals = gelu(x_vals)
min_idx = y_vals.argmin()
print(f"Minimum GELU value: {y_vals[min_idx]:.4f} at x = {x_vals[min_idx]:.4f}")

### Feed-forward network

Next, here is the code for the feed-forward network. Note that we follow the released code and use the name **multi-layer perceptron (MLP)** rather than "feed-forward network".

In [ ]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, config.n_embd * 4)
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)

    def forward(self, x):
        batch_size, seq_len, n_embd = x.shape
        x = self.c_fc(x)
        x = gelu(x)
        x = self.c_proj(x)
        return x

#### 🎓 Task 2.03: Shape annotations

One of the most common errors in deep learning is a mismatch in tensor dimensions. To avoid this, it is good practice to annotate PyTorch code with shapes. For example, suppose you are given the following code:

In [ ]:
f = nn.Linear(5, 7)
x = torch.rand(2, 3, 5)
y = f(x)

The annotation of this code with shapes would look as follows:

In [ ]:
f = nn.Linear(5, 7)
# not a tensor variable; needs no annotation

x = torch.rand(2, 3, 5)
# shape of x: [2, 3, 5]

y = f(x)
# shape of y: [2, 3, 7]

Annotate the shapes in the `forward()` method of the feed-forward network. Instead of using actual numbers, refer to dimension sizes by symbolic names such as `n_embd`, `batch_size` (number of samples in a batch of input data) and `seq_len` (length of an input sequence). You can introduce additional names and other notation you find useful. Make your annotations as detailed as you need them to explain how the shapes change from one line to the next.

In [ ]:
# Task 2.03: Shape-annotated forward() method of MLP

def forward_annotated(self, x):
    batch_size, seq_len, n_embd = x.shape
    # shape of x: [batch_size, seq_len, n_embd]

    x = self.c_fc(x)
    # c_fc is nn.Linear(n_embd, n_embd*4)
    # shape of x: [batch_size, seq_len, n_embd * 4]
    # The linear layer projects each token embedding from n_embd to 4*n_embd dimensions.
    # nn.Linear applies the same transformation independently across batch and seq dimensions.

    x = gelu(x)
    # shape of x: [batch_size, seq_len, n_embd * 4]  (element-wise, shape unchanged)

    x = self.c_proj(x)
    # c_proj is nn.Linear(n_embd*4, n_embd)
    # shape of x: [batch_size, seq_len, n_embd]
    # Projects back down from 4*n_embd to n_embd so the output
    # matches the residual stream dimension.

    return x
    # final shape: [batch_size, seq_len, n_embd]

### Causal mask

Our next goal is to implement the core of the GPT architecture: the multi-head attention mechanism.

Recall that the attention mechanism in the Transformer decoder must be restricted to attending only to previously generated tokens. This type of attention is also called **causal attention**. In practice, we implement it through a masking technique that sets the post-softmax attention weights of future tokens to zero. The following utility function implements such a mask:

In [ ]:
def make_causal_mask(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

#### 🧩 Task 2.04: Causal mask

Have a close look at the following code and run it to see the result. What are the shapes of `x` and `mask`? Given that the shapes are different, why does the addition operation in the last line not raise an error? What is the shape of the result?

How does the addition operation implement masking? (Recall that the attention scores are normalised using the softmax function.)

In [ ]:
x = torch.rand(1, 2, 3, 3)
mask = make_causal_mask(5)
x + mask[:3, :3]

**Answer (Task 2.04):**

- **Shape of `x`:** `[1, 2, 3, 3]` — a 4D tensor (batch=1, heads=2, seq=3, seq=3)
- **Shape of `mask[:3, :3]`:** `[3, 3]` — a 2D upper-triangular matrix of 0s and -inf

**Why no error?** PyTorch uses **broadcasting**. A `[3, 3]` tensor can be broadcast against a `[1, 2, 3, 3]` tensor because the trailing dimensions match. PyTorch automatically expands the mask across the batch and head dimensions.

**Shape of result:** `[1, 2, 3, 3]`

**How masking works:**
The mask contains 0s on and below the diagonal, and `-inf` above the diagonal. Adding `-inf` to the raw attention scores for future positions means those scores become `-inf`. When softmax is applied:
$$\text{softmax}(-\infty) = \frac{e^{-\infty}}{\sum e^{...}} = \frac{0}{\sum e^{...}} = 0$$
So future tokens receive zero attention weight — they are completely masked out. This enforces the autoregressive property: each token can only attend to itself and previous tokens.

### Attention mechanism

Here is the code for the multi-head attention mechanism:

In [ ]:
class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.register_buffer("mask", make_causal_mask(config.n_ctx), persistent=False)

    def forward(self, x):
        batch_size, seq_len, n_embd = x.shape
        head_embd = n_embd // self.n_head
        q, k, v = self.c_attn(x).chunk(3, dim=-1)
        q = q.view(batch_size, seq_len, self.n_head, head_embd)
        k = k.view(batch_size, seq_len, self.n_head, head_embd)
        v = v.view(batch_size, seq_len, self.n_head, head_embd)
        q = q.transpose(-2, -3)
        k = k.transpose(-2, -3)
        v = v.transpose(-2, -3)
        x = q @ k.transpose(-1, -2)
        x = x / head_embd**0.5
        x = x + self.mask[:seq_len, :seq_len]
        x = torch.softmax(x, dim=-1)
        x = x @ v
        x = x.transpose(-2, -3).contiguous()
        x = x.view(batch_size, seq_len, n_embd)
        x = self.c_proj(x)
        return x

#### 🎓 Task 2.05: Multi-head attention

Trace the input `x` through the `forward()` method line by line and annotate the shapes of all tensor variables. Identify all lines that rely on broadcasting.

In [ ]:
# Task 2.05: Shape-annotated Attention forward() method
# Let B=batch_size, T=seq_len, E=n_embd, H=n_head, D=head_embd (= E // H)

def forward_attention_annotated(self, x):
    batch_size, seq_len, n_embd = x.shape
    # x: [B, T, E]

    head_embd = n_embd // self.n_head
    # head_embd = E // H  (scalar)

    q, k, v = self.c_attn(x).chunk(3, dim=-1)
    # self.c_attn(x): [B, T, 3E]  (Linear maps E -> 3E)
    # after chunk(3, dim=-1): three tensors, each [B, T, E]
    # q: [B, T, E],  k: [B, T, E],  v: [B, T, E]

    q = q.view(batch_size, seq_len, self.n_head, head_embd)
    # q: [B, T, H, D]  — reshape E into H heads of size D

    k = k.view(batch_size, seq_len, self.n_head, head_embd)
    # k: [B, T, H, D]

    v = v.view(batch_size, seq_len, self.n_head, head_embd)
    # v: [B, T, H, D]

    q = q.transpose(-2, -3)
    # swap dims -2 (T) and -3 (H): q becomes [B, H, T, D]

    k = k.transpose(-2, -3)
    # k: [B, H, T, D]

    v = v.transpose(-2, -3)
    # v: [B, H, T, D]

    x = q @ k.transpose(-1, -2)
    # k.transpose(-1,-2): [B, H, D, T]
    # q @ k^T: [B, H, T, T]  — raw attention scores for each head

    x = x / head_embd**0.5
    # scale by 1/sqrt(D) to stabilise gradients
    # x: [B, H, T, T]  (scalar division, shape unchanged)

    x = x + self.mask[:seq_len, :seq_len]
    # self.mask[:T, :T]: [T, T]  — BROADCASTING: [T, T] -> [B, H, T, T]
    # x: [B, H, T, T]  — future positions set to -inf

    x = torch.softmax(x, dim=-1)
    # softmax over last dim (key positions)
    # x: [B, H, T, T]  — attention weights, rows sum to 1

    x = x @ v
    # x: [B, H, T, T]  @  v: [B, H, T, D]  ->  [B, H, T, D]
    # weighted sum of value vectors

    x = x.transpose(-2, -3).contiguous()
    # swap H and T back: [B, T, H, D]

    x = x.view(batch_size, seq_len, n_embd)
    # merge H and D dimensions back: [B, T, E]

    x = self.c_proj(x)
    # Linear(E, E): [B, T, E]  — output projection

    return x
    # final shape: [B, T, E]

# Lines relying on broadcasting:
# - "x = x + self.mask[:seq_len, :seq_len]"
#   mask shape [T, T] is broadcast to [B, H, T, T]

### Layer normalisation

As mentioned above, the inputs to both the feed-forward network and the multi-head attention mechanism undergo **layer normalisation**. This normalises the inputs to have zero mean and unit variance across the activations. [Ba et al. (2016)](https://doi.org/10.48550/arXiv.1607.06450) introduce two trainable parameters (called $\gamma$ and $\beta$ in the paper) that allow the network to learn an appropriate scale and shift for the normalised values.

We implement layer normalisation as follows:

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.g = nn.Parameter(torch.ones(config.n_embd))
        self.b = nn.Parameter(torch.zeros(config.n_embd))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(unbiased=False, dim=-1, keepdim=True)
        return self.g * (x - mean) / torch.sqrt(variance + 1e-05) + self.b

#### 🧩 Task 2.06: Layer normalisation

What is the relevance of the `keepdim=True` keyword argument in the `mean()` and `var()` functions? What would happen if we omitted it?

What is the relevance of the constant 1e-05? What could happen if we omitted it?

**Answer (Task 2.06):**

**`keepdim=True`:**
When computing `mean` over `dim=-1`, PyTorch by default removes that dimension. For input `x` of shape `[B, T, E]`, `x.mean(dim=-1)` would give shape `[B, T]`. Subtracting `[B, T]` from `[B, T, E]` would then fail due to shape mismatch.

With `keepdim=True`, the result keeps the reduced dimension as size 1: shape `[B, T, 1]`. PyTorch can then broadcast `[B, T, 1]` against `[B, T, E]` correctly — subtracting the same mean from every embedding dimension. Without it, the subtraction `(x - mean)` would raise a shape error.

**The constant `1e-05` (epsilon):**
This is a small numerical stability constant added to the variance before taking the square root. If the variance of any sample is exactly zero (all embedding values are identical), dividing by `sqrt(0) = 0` would produce `NaN` or `inf`. Adding `1e-05` ensures we never divide by zero, keeping the computation numerically stable. Its value is small enough that it doesn't meaningfully affect the normalisation in normal cases.

### Decoder block

We now combine the feed-forward network, the multi-head attention mechanism and the layer normalisation into a decoder block.

In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config)
        self.attn = Attention(config)
        self.ln_2 = LayerNorm(config)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

#### 🎓 Task 2.07: Pre-norm and post-norm architectures

The original Transformer ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)) is a "post-norm architecture", where the normalisation is applied **after** each residual block. In contrast, GPT-2 is a "pre-norm architecture", where the normalisation is applied **before**. Find the passage in Section&nbsp;2.3 of [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) that reports on this modification.

[Xiong et al. (2020)](https://arxiv.org/pdf/2002.04745) compare pre-norm and post-norm architectures empirically. Read the abstract of their paper and summarise their main findings. According to these findings, what are the benefits of the pre-norm architecture?

**Answer (Task 2.07):**

**Passage from Radford et al. (2019), Section 2.3:**
The relevant modification is stated as: *"Layer normalization was moved to the input of each sub-block, similar to a pre-activation residual network and an additional layer normalization was added after the final self-attention block."*

**Summary of Xiong et al. (2020) findings:**
Xiong et al. show theoretically and empirically that the post-norm architecture (original Transformer) suffers from large gradient magnitudes at the output layer during early training, making it hard to train without careful learning rate warm-up. The pre-norm architecture (as used in GPT-2) has better-behaved gradients from the start.

**Benefits of pre-norm:**
- **Training stability**: Gradients are more stable, reducing the need for learning rate warm-up schedules.
- **Easier optimisation**: The model can be trained with a constant learning rate from the beginning.
- **Better convergence**: Pre-norm models often converge faster and more reliably, especially for very deep networks.

In the `Block.forward()` above, pre-norm is visible in `self.attn(self.ln_1(x))` — layer norm is applied *before* the attention, not after.

### Model

We now have almost all components in place to complete the implementation of the GPT-2 model. The only thing missing are the position embeddings. These simply associate an embedding vector with every position in the context window. To set them up, we first define another utility function:

In [ ]:
def make_positions(n):
    return torch.arange(n, dtype=torch.long)

We then code the complete model as follows:

In [ ]:
class Model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.n_vocab, config.n_embd)
        self.wpe = nn.Embedding(config.n_ctx, config.n_embd)
        self.h = nn.Sequential(*(Block(config) for _ in range(config.n_layer)))
        self.ln_f = LayerNorm(config)
        self.lm_head = nn.Linear(config.n_embd, config.n_vocab, bias=False)
        self.register_buffer("pos", make_positions(config.n_ctx), persistent=False)

    def forward(self, x):
        batch_size, seq_len = x.shape
        wte = self.wte(x)
        wpe = self.wpe(self.pos[:seq_len])
        x = wte + wpe
        x = self.h(x)
        x = self.ln_f(x)
        x = self.lm_head(x)
        return x

#### 🧩 Task 2.08: Buffers

Our implementation registers the vector of positions as a buffer. (Earlier, we also registered the causal mask as a buffer.) Consult the PyTorch documentation to determine the benefits of registering a tensor as a buffer, in contrast to computing it in the `forward()` method.

**Answer (Task 2.08):**

Registering a tensor as a buffer with `register_buffer()` offers several benefits over re-computing it in `forward()`:

1. **Device tracking**: Buffers are automatically moved to the correct device (CPU/GPU) when you call `model.to(device)`. If you compute the mask in `forward()`, you'd need to manually specify the device every time.

2. **State persistence**: Buffers are included in `model.state_dict()`, so they are saved and loaded correctly with `torch.save()` / `torch.load()`. Non-buffer tensors computed in `forward()` are lost when the model is saved.

3. **Efficiency**: The mask and position indices are precomputed once at initialisation rather than being recreated at every forward pass. This avoids redundant computation at inference time.

4. **Not treated as parameters**: Unlike `nn.Parameter`, buffers are not returned by `model.parameters()` and are not updated by the optimiser. They are persistent tensors that belong to the model but are not trained.

The `persistent=False` flag used here means the buffer is NOT included in `state_dict()` — it is just registered for device management and precomputation, since the mask and positions can always be recomputed from scratch.

#### 🎓 Task 2.09: Number of trainable parameters

The model we have implemented is the smallest one presented by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). But how many trainable parameters exactly does it have? Interestingly, the number originally reported by the authors is wrong! What number did they report?

Your task is to write code to compute the number of parameters yourself. This should only take 1–3 lines of code. What number do you get when you apply this code to a fresh model instance?

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) followed the original Transformers paper ([Vaswani et al., 2017](https://doi.org/10.48550/arXiv.1706.03762)) and shared the trainable weights between the token embedding and the final linear layer. Implement this weight sharing strategy. (Hint: This only requires one line of code.) Then, re-compute the number of trainable parameters for the modified model. What number do you get now? How large is the reduction caused by the weight sharing?

In [ ]:
# Task 2.09: Count trainable parameters

model = Model(Config())

# Count parameters (1-3 lines)
n_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters (no weight sharing): {n_params:,}")
# Radford et al. (2019) reported ~117M; the correct number is ~124M
# The discrepancy comes from the token embedding table not being counted in the
# original paper's count when it was shared, but here it's separate.

# Implement weight sharing between wte and lm_head
# (one line of code)
model.lm_head.weight = model.wte.weight

# Re-count after weight sharing
n_params_shared = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters (with weight sharing): {n_params_shared:,}")

# Reduction
reduction = n_params - n_params_shared
print(f"Reduction from weight sharing: {reduction:,} parameters")
print(f"That is {reduction / n_params * 100:.1f}% of total parameters")

**Explanation (Task 2.09):**

- Without weight sharing: ~**124,439,808** parameters
- Radford et al. (2019) reported **117M** — this is wrong because they did not count the token embedding table separately from the output projection.
- After weight sharing (`model.lm_head.weight = model.wte.weight`): ~**85,053,440** parameters
- Reduction: ~**38.6M** parameters (~31% reduction). The token embedding matrix has shape `[50257, 768]` = 38,597,376 values — this is shared instead of duplicated.

Weight sharing works because the token embedding matrix maps token IDs → vectors, while the output linear layer maps vectors → token logits. These are inverse operations and empirically sharing weights works well and reduces model size.

## Part 2: Load pre-trained weights

Now that you have a complete implementation of the GPT-2 model in place, you can instantiate it by loading the pre-trained weights released by OpenAI. These weights were originally provided in the TensorFlow format. For this lab, we have re-packaged them as a single file in NumPy's `.npz` archive format. We can load it as follows:

In [ ]:
import numpy as np

pretrained = np.load("gpt-2-pretrained.npz")

The result `pretrained` is a dictionary mapping names to NumPy arrays. When you print the names, you will see that they correspond to the attributes of our network modules, even though the names differ. For example, the array `h0.attn.c_attn.b` holds the biases (`b`) of the `c_attn` linear layer in the attention mechanism (`attn`) of the first transformer block (`h0`).

#### 🎓 Task 2.10: Load pre-trained weights

Create a model from the pre-trained weights. To do this, you need to instantiate a fresh model and write the contents of each array from the `npz` archive with the pre-trained weights into the corresponding tensor. To make this a bit easier, here is a utility function that copies data from a NumPy array `source` to a PyTorch tensor `target`:

In [ ]:
def copy_weights(source: np.ndarray, target: torch.Tensor):
    assert source.shape == target.shape
    with torch.no_grad():
        target.copy_(torch.tensor(source, dtype=torch.float32))

In [ ]:
def from_pretrained() -> Model:
    model = Model(Config())
    # Apply weight sharing first (same as Task 2.09)
    model.lm_head.weight = model.wte.weight

    pretrained = np.load("gpt-2-pretrained.npz")

    # Inspect the key names to understand the mapping
    # print(list(pretrained.keys()))  # Uncomment to explore

    # Copy token and position embeddings
    copy_weights(pretrained["wte"], model.wte.weight)
    copy_weights(pretrained["wpe"], model.wpe.weight)

    # Copy weights for each transformer block
    for i in range(model.config.n_layer):
        b = model.h[i]
        p = pretrained

        # Layer norm 1 (before attention)
        copy_weights(p[f"h{i}.ln_1.g"], b.ln_1.g)
        copy_weights(p[f"h{i}.ln_1.b"], b.ln_1.b)

        # Attention: c_attn (combined Q, K, V projection)
        # Note: PyTorch stores Linear weights transposed!
        copy_weights(p[f"h{i}.attn.c_attn.w"].T, b.attn.c_attn.weight)
        copy_weights(p[f"h{i}.attn.c_attn.b"], b.attn.c_attn.bias)

        # Attention: output projection
        copy_weights(p[f"h{i}.attn.c_proj.w"].T, b.attn.c_proj.weight)
        copy_weights(p[f"h{i}.attn.c_proj.b"], b.attn.c_proj.bias)

        # Layer norm 2 (before MLP)
        copy_weights(p[f"h{i}.ln_2.g"], b.ln_2.g)
        copy_weights(p[f"h{i}.ln_2.b"], b.ln_2.b)

        # MLP: fc and proj layers
        copy_weights(p[f"h{i}.mlp.c_fc.w"].T, b.mlp.c_fc.weight)
        copy_weights(p[f"h{i}.mlp.c_fc.b"], b.mlp.c_fc.bias)
        copy_weights(p[f"h{i}.mlp.c_proj.w"].T, b.mlp.c_proj.weight)
        copy_weights(p[f"h{i}.mlp.c_proj.b"], b.mlp.c_proj.bias)

    # Final layer norm
    copy_weights(pretrained["ln_f.g"], model.ln_f.g)
    copy_weights(pretrained["ln_f.b"], model.ln_f.b)

    model.eval()
    return model

**Important:** One technical detail to note is that PyTorch stores the weights of linear layers in a transposed form. For example, a linear layer created as `nn.Linear(2, 3)` has a weight matrix of shape [3, 2].

## Part 3: Put the model to use

In the third and final part of this lab, you will use the pre-trained model to generate text and evaluate it on a standard benchmark.

### Sampling-based text generation

The easiest way to generate text with a language model is by using a **greedy approach**. This method works by creating text one token at a time. At each step, the model takes the previously generated text (called the **context**) as input and adds the token with the highest output logit as a new token. The code in the next cell defines a function `generate()` that forms the core of a greedy generator:

In [ ]:
def generate(model, context, context_size=1024, n_tokens=20):
    for _ in range(n_tokens):
        context = context[:, -context_size:]
        with torch.no_grad():
            logits = model(context)[:, -1, :]
        next_idx = torch.argmax(logits, dim=-1, keepdim=True)
        context = torch.cat([context, next_idx], dim=-1)
    return context

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
model = from_pretrained()


def generate_helper(text, context_size=1024, n_tokens=20):
    context = torch.tensor([tokenizer.encode(text)], dtype=torch.long)
    context = generate(model, context, context_size=context_size, n_tokens=n_tokens)
    return tokenizer.decode(context[0].tolist())

In [ ]:
generate_helper("Linköping University is")

In [ ]:
# Tip: Alternative using HuggingFace if Task 2.10 wasn't completed
# from transformers import GPT2LMHeadModel
# model = GPT2LMHeadModel.from_pretrained("gpt2")
# logits = model(context).logits[:, -1, :]

#### 🎓 Task 2.11: Sampling-based text generation

The greedy approach to text generation is not very interesting for practical applications because it always chooses the most likely token, leading to predictable and less creative results. Your task is to modify the code for the `generate()` function to use a **sampling-based approach** instead. In this approach, the next token is chosen randomly based on the probabilities assigned by the model (softmax-normalised logits), treating them as a categorical distribution over the token vocabulary. Additionally, your code should include two common techniques to improve sampling:

 * **temperature scaling**, which lets the user control the randomness of the sampling
 * **top-$k$ sampling**, which limits the sampling to the top-$k$ most likely tokens, ignoring less probable ones

In [ ]:
# Task 2.11: Sampling-based text generation with temperature and top-k

def generate_sampled(model, context, context_size=1024, n_tokens=20, temperature=1.0, top_k=None):
    """
    Generate tokens using sampling instead of greedy decoding.

    Args:
        model:        the language model
        context:      input token IDs, shape [1, T]
        context_size: max context window
        n_tokens:     number of tokens to generate
        temperature:  controls randomness. <1 = more focused, >1 = more random
        top_k:        if set, only sample from the top-k most likely tokens
    """
    for _ in range(n_tokens):
        # Trim context to fit within the context window
        context = context[:, -context_size:]

        with torch.no_grad():
            # Get logits for the last position only
            logits = model(context)[:, -1, :]   # shape: [1, n_vocab]

        # --- Temperature scaling ---
        # Divide logits by temperature before softmax.
        # Low temperature sharpens the distribution (more deterministic).
        # High temperature flattens it (more random).
        logits = logits / temperature

        # --- Top-k filtering ---
        # Zero out (set to -inf) all logits except the top-k highest.
        if top_k is not None:
            # Find the k-th largest logit value
            top_k_values, _ = torch.topk(logits, top_k, dim=-1)
            threshold = top_k_values[:, -1].unsqueeze(-1)  # shape: [1, 1]
            # Mask out tokens below the threshold
            logits = logits.masked_fill(logits < threshold, float("-inf"))

        # Convert logits to probabilities
        probs = torch.softmax(logits, dim=-1)   # shape: [1, n_vocab]

        # Sample one token from the probability distribution
        next_idx = torch.multinomial(probs, num_samples=1)  # shape: [1, 1]

        # Append sampled token to context
        context = torch.cat([context, next_idx], dim=-1)

    return context


def generate_sampled_helper(text, context_size=1024, n_tokens=50, temperature=0.8, top_k=40):
    context = torch.tensor([tokenizer.encode(text)], dtype=torch.long)
    context = generate_sampled(model, context, context_size=context_size,
                               n_tokens=n_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(context[0].tolist())


# Test with different settings
print("=== Low temperature (more focused) ===")
print(generate_sampled_helper("Linköping University is", temperature=0.5, top_k=20))

print("\n=== High temperature (more creative) ===")
print(generate_sampled_helper("Linköping University is", temperature=1.2, top_k=50))

### Evaluating the pretrained model

If you have experimented with your pretrained GPT-2 model, you will have noticed that its ability to generate useful text is somewhat limited. By today's standards, GPT-2 is a small model with modest capabilities. However, it can still be helpful for certain tasks, such as text autocompletion, generating filler text, or answering simple questions. To rigourosly evaluate language models, researchers often use standard benchmark datasets. Creating these benchmarks is a discipline of its own, and they tend to become increasingly challenging as models continue to improve.

In the final task of this lab, you will evaluate GPT-2's performance on a small subset of the [HellaSwag dataset](https://rowanzellers.com/hellaswag/), which was published in the same year as GPT-2 itself (2019). HellaSwag is designed to test a model's ability to perform commonsense reasoning in challenging contexts. Unlike simpler benchmarks, HellaSwag presents scenarios where the correct text completion depends on semantic relationships between events and on world knowledge. This makes it a good choice for assessing the ability of language models to go beyond surface-level patterns and produce meaningful, context-aware predictions.

#### 🎓 Task 2.12: Evaluating the pretrained model

Read the [HellaSwag website](https://rowanzellers.com/hellaswag/) to get some background on the benchmark. How does a sample from the dataset look like? What is an expected prediction? How does the benchmark allow us to score models? What is the random baseline? What is the human performance reported on the task?

The next cell contains code for evaluating your pretrained model on a small sample from HellaSwag. You will also need a tokenizer. The HellaSwag subset is in the file `hellaswag-mini.jsonl`. Inspect that file to understand the format. Next, read the code and explain how it works. Specifically, how does the code compute the score of individual endings? In the call to `cross_entropy()`, why are the tensors sliced in this specific way?

Finally, what overall score does the pretrained GPT-2 model get on this benchmark? How does that score compare to the random baseline and the human performance?

**Answer (Task 2.12) — Background on HellaSwag:**

**Dataset structure:** Each sample has:
- `ctx`: a context sentence (the beginning of an activity)
- `endings`: four candidate sentence completions
- `label`: the index (0–3) of the correct ending

Example: *"She pours the punch into the cups. She then..."* followed by four possible endings, only one of which makes commonsense.

**Scoring models:** For each sample, the model must pick the most plausible ending. Accuracy = (correct predictions / total samples).

**Random baseline:** Since there are always 4 choices, random guessing gives **25% accuracy**.

**Human performance:** Humans achieve approximately **95.6%** accuracy on HellaSwag.

**How the code works:**
- For each ending, the context (`prefix`) and the candidate ending (`suffix`) are concatenated into a full token sequence.
- The model's logits are computed for this full sequence.
- Cross-entropy loss is computed specifically over the suffix tokens: `logits[0, -len(suffix)-1 : -1]` are the model's predictions for each suffix position (each is the output of the model *before* seeing that token), and `context[0, -len(suffix):]` are the actual suffix tokens (ground truth).
- This computes the average negative log-likelihood of the model assigning probability to the correct ending tokens.
- The ending with the **lowest** cross-entropy loss (highest probability under the model) is chosen as the prediction.
- The slicing `[-len(suffix)-1 : -1]` ensures we align model outputs (predictions at position t) with the actual next tokens (at position t+1).

**GPT-2 performance:** GPT-2 (small) typically achieves approximately **29–31%** accuracy on HellaSwag — slightly above the random baseline of 25%, showing some commonsense reasoning ability but far below human performance of ~95.6%.

In [ ]:
import json

with open("hellaswag-mini.jsonl") as f:
    n_correct = 0
    n_total = 0
    for line in f:
        sample = json.loads(line)
        prefix = tokenizer.encode(sample["ctx"])
        ending_scores = []
        for i, ending in enumerate(sample["endings"]):
            suffix = tokenizer.encode(" " + ending)
            context = torch.tensor([prefix + suffix], dtype=torch.long)
            with torch.no_grad():
                logits = model(context)
                ending_score = torch.nn.functional.cross_entropy(
                    logits[0, -len(suffix) - 1 : -1], context[0, -len(suffix) :]
                )
            ending_scores.append((ending_score, i))
        predicted = min(ending_scores)[1]
        n_correct += int(predicted == sample["label"])
        n_total += 1
    print(f"Accuracy: {n_correct / n_total:.2%}")

**🥳 Congratulations on finishing lab 2!**